In [ ]:
#Cell 1 - Imports

import requests
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection

print("Imports Successful")

In [ ]:
#Cell 2 - Config

BLOB_BASE = "https://YOUR_BLOB_STORAGE_ACCOUNT.blob.core.windows.net/YOUR_BLOB_CONTAINER/StationPhotos/"
FEATURE_LAYER_ITEM_ID = "YOUR FEATURE LAYER ITEM ID"
MAX_PHOTOS = 20  # maximum number of photos to probe per station

print("Configured")

In [ ]:
#Cell 3 - Connect to AGOL

gis = GIS("home")
print(f"Signed in as: {gis.properties.user.username}")
item = gis.content.get(FEATURE_LAYER_ITEM_ID)
flc = FeatureLayerCollection.fromitem(item)
layer = flc.layers[0]
print("Layer connected.")

In [ ]:
#4 - Query all station codes from layer

features = layer.query(out_fields="OBJECTID,m_StationCode,photo_url,photo_count").features
print(f"  → {len(features)} stations found in layer")

In [ ]:
#Cell 5 - Probe blob for photos per station (gap-tolerant)

def count_photos(code):
    """Probe blob for all photos up to MAX_PHOTOS, tolerating gaps."""
    urls = []
    for i in range(1, MAX_PHOTOS + 1):
        url = f"{BLOB_BASE}{code}{i:02d}.jpg"
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            urls.append(url)
    return len(urls), urls

print("Probing blob for photos...")
results = {}
for f in features:
    code = f.attributes.get("m_StationCode")
    if code:
        count, urls = count_photos(code)
        results[code] = {"count": count, "urls": urls}
        if count > 0:
            print(f"  {code}: {count} photo(s) found")

print(f"\nDone probing. {sum(1 for v in results.values() if v['count'] > 0)} stations have photos.")

In [ ]:
#Cell 6 - Update photo_url JSON array and photo_count in feature layer

import json

updates = []
for f in features:
    code = f.attributes.get("m_StationCode")
    if code and code in results:
        new_count = results[code]["count"]
        new_url = results[code]["urls"][0] if results[code]["urls"] else None
        current_count = f.attributes.get("photo_count")
        current_url = f.attributes.get("photo_url")

        if new_count != current_count or new_url != current_url:
            updates.append({
                "attributes": {
                    "OBJECTID": f.attributes["OBJECTID"],
                    "photo_url": new_url,
                    "photo_count": new_count,
                }
            })

if updates:
    result = layer.edit_features(updates=updates)
    print(f"  → {len(updates)} stations updated")
else:
    print("  → No changes needed")

print("Done!")

In [ ]:
# Cell 7: Build and store photo HTML per station
base = "https://YOUR_BLOB_STORAGE_ACCOUNT.blob.core.windows.net/YOUR_BLOB_CONTAINER/StationPhotos/"
html_updates = []

for f in features:
    code = f.attributes.get("m_StationCode")
    if code and code in results and results[code]["count"] > 0:
        count = results[code]["count"]
        html = "<div style='width:100%;'>"
        for i in range(1, count + 1):
            num = f"{i:02d}"
            html += f"<img src='{base}{code}{num}.jpg' style='width:100%;border-radius:4px;margin-bottom:4px;'>"
        html += "</div>"
        html_updates.append({
            "attributes": {
                "OBJECTID": f.attributes["OBJECTID"],
                "photo_html": html,
            }
        })

if html_updates:
    result = layer.edit_features(updates=html_updates)
    print(f"  → {len(html_updates)} stations photo HTML updated")
else:
    print("  → No updates needed")
print("Done!")